# Sumativa 3 · Taller II — Integración de fuentes heterogéneas

**Universidad:** Universidad Andrés Bello · Magíster en Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Gestión de Datos y Tecnologías Big Data (MCDI502) · **Grupo 7**  
**Integrantes:** Cristian Grandón, Felipe Palma y Jorge Gutiérrez  
**Entrega:** `mcdi502_s3_grupo7.ipynb` + video grupal de 7–10 minutos en un único ZIP.

**Objetivo:** crear cuatro DataFrames Spark, uno por fuente (Open-Meteo, AQICN, MongoDB ESP32 y MySQL/REST), de **100 registros reales cada uno**, calcular dos columnas, integrar las cuatro fuentes con una estrategia justificada y ejecutar las consultas **C1–C7 usando Spark SQL**.

> **Reproducibilidad:** este notebook no contiene resultados ni credenciales inventadas. Depende de los servicios externos y del acceso entregado por el docente. La ingesta y los controles señalan expresamente si alguna fuente no alcanzó las 100 filas exigidas; antes de entregar es obligatorio ejecutar todo, guardar las salidas y resolver los pendientes.

## 1. Preparación e instalación del entorno — configuración inicial (6 pts)

**Recomendado:** Google Colab (las instrucciones lo citan como ejemplo, no como obligación). En Colab se instalan las dependencias si faltan. En VS Code + Jupyter también funciona, siempre que tengas Java JDK 17/21 y ejecutes previamente `python -m pip install pyspark==4.0.3 requests pymongo` en el entorno activo.

- `SparkSession` y `SparkContext`: motor de procesamiento distribuido.
- `requests`: Open-Meteo, AQICN y MySQL vía API REST.
- `pymongo`: conexión real con MongoDB sin depender de un JAR externo de versión incompatible.
- Credenciales: se solicitan durante la ejecución y **no se guardan** en GitHub.

**Importante:** si instalas o cambias PySpark en un kernel que ya tenía Spark importado, reinicia ese kernel antes de continuar.

In [1]:
import sys, os, shutil, subprocess, importlib.util
from importlib import metadata

EN_COLAB = 'google.colab' in sys.modules
PAQUETES = {'pyspark': '4.0.3', 'requests': None, 'pymongo': None, 'dnspython': None}
pendientes = []
for paquete, version in PAQUETES.items():
    try:
        instalada = metadata.version(paquete)
        if version and instalada != version:
            pendientes.append(f'{paquete}=={version}')
    except metadata.PackageNotFoundError:
        pendientes.append(f'{paquete}=={version}' if version else paquete)

if pendientes:
    if EN_COLAB:
        print('Instalando dependencias:', pendientes)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pendientes])
    else:
        raise RuntimeError('Instala en tu venv y reinicia Jupyter: python -m pip install ' + ' '.join(pendientes))

if shutil.which('java') is None:
    if EN_COLAB:
        subprocess.check_call(['apt-get', 'update', '-qq'])
        subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'])
    else:
        raise RuntimeError('Java no está en PATH. Instala JDK 17 o 21 y reinicia VS Code.')

print('Python:', sys.version.split()[0], '| Java:', shutil.which('java'))

Python: 3.14.8 | Java: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot\bin\java.EXE


### Inicialización de Spark

Se configura la zona horaria `America/Santiago` para que las fechas usadas en C2 y C3 sean consistentes. El procesamiento se realiza en un motor local Spark, suficiente para demostrar las operaciones de esta actividad.

In [2]:
import time
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from getpass import getpass
from urllib.parse import quote_plus
import requests
from pymongo import MongoClient, DESCENDING
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

ZONA_HORARIA = 'America/Santiago'
ZONA = ZoneInfo(ZONA_HORARIA)
CANTIDAD_REQUERIDA = 100
spark = (SparkSession.builder
    .appName('MCDI502_Sumativa3_Grupo7')
    .master('local[*]')
    .config('spark.sql.session.timeZone', ZONA_HORARIA)
    .config('spark.sql.shuffle.partitions', '5')
    .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel('ERROR')
print('Apache Spark:', spark.version)
print('SparkSession:', spark.sparkContext.appName)
print('SparkContext operativo, particiones por defecto:', sc.defaultParallelism)
assert spark.range(1).count() == 1

Apache Spark: 4.0.3
SparkSession: MCDI502_Sumativa3_Grupo7
SparkContext operativo, particiones por defecto: 16


## 2. Ingesta 1/4: Open-Meteo API → `df_openmeteo`

Se capturan las variables exigidas (temperatura y humedad a 2 m, precipitación, velocidad/dirección del viento a 10 m e índice UV) y **100 observaciones horarias reales** de los últimos siete días. Por defecto se usa **Viña del Mar** siguiendo el ejemplo del grupo; se pueden ajustar coordenadas.

La muestra selecciona horas repartidas uniformemente por el período de 7 días, para que C3 no dependa solamente de las últimas 100 horas.

In [3]:
LATITUD, LONGITUD = -33.02457, -71.55183  # Viña del Mar; ajustar si corresponde
CAMPOS_METEO = [
    'temperature_2m', 'relative_humidity_2m', 'precipitation',
    'wind_speed_10m', 'wind_direction_10m', 'uv_index'
]
AHORA = datetime.now(ZONA)
INICIO_SEMANA = AHORA - timedelta(days=7)
r = requests.get('https://api.open-meteo.com/v1/forecast', params={
    'latitude': LATITUD, 'longitude': LONGITUD,
    'hourly': ','.join(CAMPOS_METEO),
    'past_days': 7, 'forecast_days': 1,
    'timezone': ZONA_HORARIA
}, timeout=35)
r.raise_for_status()
contenido = r.json()
if 'hourly' not in contenido:
    raise ValueError('Open-Meteo no entregó datos horarios: ' + str(contenido)[:200])
serie = contenido['hourly']
observaciones = []
for i, txt in enumerate(serie['time']):
    momento = datetime.fromisoformat(txt).replace(tzinfo=ZONA)
    if INICIO_SEMANA <= momento <= AHORA:
        fila = {'fecha_hora_txt': txt}
        for campo in CAMPOS_METEO:
            dato = serie.get(campo, [None] * len(serie['time']))[i]
            fila[campo] = float(dato) if dato is not None else None
        observaciones.append(fila)
if len(observaciones) < CANTIDAD_REQUERIDA:
    raise ValueError(f'Open-Meteo solo devolvió {len(observaciones)} horas de la última semana.')
indices = [round(i * (len(observaciones) - 1) / (CANTIDAD_REQUERIDA - 1)) for i in range(CANTIDAD_REQUERIDA)]
muestra_meteo = [observaciones[i] for i in indices]
esquema_meteo = T.StructType([
    T.StructField('fecha_hora_txt', T.StringType(), False),
    *[T.StructField(c, T.DoubleType(), True) for c in CAMPOS_METEO]
])
df_openmeteo = (spark.createDataFrame(muestra_meteo, schema=esquema_meteo)
    .withColumn('fecha_hora', F.to_timestamp('fecha_hora_txt', "yyyy-MM-dd'T'HH:mm"))
    .drop('fecha_hora_txt'))
print('Open-Meteo:', df_openmeteo.count(), 'registros | periodo:', INICIO_SEMANA.date(), 'a', AHORA.date())
df_openmeteo.show(5, truncate=False)

Open-Meteo: 100 registros | periodo: 2026-11-07 a 2026-11-14
+--------------+--------------------+-------------+--------------+------------------+--------+-------------------+
|temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|fecha_hora         |
+--------------+--------------------+-------------+--------------+------------------+--------+-------------------+
|17.4          |64.0                |0.0          |8.1           |162.0             |0.0     |2026-11-07 23:00:00|
|16.3          |67.0                |0.0          |6.9           |150.0             |0.0     |2026-11-08 01:00:00|
|15.8          |69.0                |0.0          |6.1           |148.0             |0.0     |2026-11-08 02:00:00|
|14.6          |76.0                |0.0          |5.6           |151.0             |0.0     |2026-11-08 04:00:00|
|13.8          |81.0                |0.0          |5.8           |154.0             |0.0     |2026-11-08 06:00:00|
+--------------+---

## 3. Ingesta 2/4: AQICN API → `df_aqicn`

**Requiere un token AQICN válido.** Se consulta la estación indicada por `CIUDAD_AQICN`. Por defecto es Santiago; si la pauta de tu grupo exige otra región, modifica esta variable **antes de ejecutar**.

Se realizan **100 solicitudes independientes** y se registran sus respuestas (el código nunca inventa filas). AQICN puede devolver **el mismo dato de medición** en varias solicitudes: 100 capturas **no son necesariamente 100 mediciones históricas diferentes**. El control final informa cuántas fechas de medición distintas hubo. Si el token falla o la región no corresponde, se detiene la ejecución con un mensaje explícito. Los valores `iaqi` son índices individuales AQI, **no** concentraciones en µg/m³.

In [4]:
CIUDAD_AQICN = os.getenv('AQICN_CIUDAD', 'santiago').strip().lower()
TOKEN_AQICN = os.getenv('AQICN_TOKEN') or getpass('Token AQICN (no se guardará): ').strip()
if not TOKEN_AQICN:
    raise RuntimeError('Se necesita un token AQICN válido para completar la ingesta.')

URL_AQICN = f'https://api.waqi.info/feed/{CIUDAD_AQICN}/'
CLAVES_AQICN = {'pm25': 'PM2_5', 'pm10': 'PM10', 'o3': 'O3', 'no2': 'NO2', 'so2': 'SO2'}
def valor_iaqi(iaqi, nombre):
    dato = iaqi.get(nombre, {})
    try:
        return float(dato.get('v')) if dato.get('v') is not None else None
    except (ValueError, TypeError, AttributeError):
        return None

filas_aqicn = []
for intento in range(CANTIDAD_REQUERIDA):
    respuesta = requests.get(URL_AQICN, params={'token': TOKEN_AQICN}, timeout=30)
    respuesta.raise_for_status()
    cuerpo = respuesta.json()
    if cuerpo.get('status') != 'ok':
        raise RuntimeError(f'AQICN rechazó captura {intento+1}: {cuerpo.get("data")}')
    estacion = cuerpo['data'].get('city', {}).get('name', '')
    if intento == 0:
        print('Estación AQICN:', estacion)
        print('Coordenadas:', cuerpo['data'].get('city', {}).get('geo'))
    coordenadas = cuerpo['data'].get('city', {}).get('geo') or []
    coincide_nombre = CIUDAD_AQICN in estacion.casefold()
    # Para Santiago se admite también una estación periférica que informe coordenadas
    # dentro del área metropolitana, aunque su nombre sea un barrio/comuna.
    coincide_coordenadas = (
        CIUDAD_AQICN == 'santiago' and len(coordenadas) == 2
        and -34.4 <= float(coordenadas[0]) <= -32.7
        and -71.9 <= float(coordenadas[1]) <= -69.7
    )
    if not (coincide_nombre or coincide_coordenadas):
        raise ValueError(f'La estación AQICN ({estacion}; {coordenadas}) no coincide con la región solicitada ({CIUDAD_AQICN}). Revisa el token.')
    iaqi = cuerpo['data'].get('iaqi', {})
    fila = {
        'numero_captura': intento + 1,
        'estacion': estacion,
        'fecha_medicion': str(cuerpo['data'].get('time', {}).get('iso') or cuerpo['data'].get('time', {}).get('s') or ''),
        'momento_captura': datetime.now(ZONA).isoformat(timespec='seconds'),
    }
    fila.update({salida: valor_iaqi(iaqi, entrada) for entrada, salida in CLAVES_AQICN.items()})
    filas_aqicn.append(fila)
    if intento < CANTIDAD_REQUERIDA - 1:
        time.sleep(1)  # Moderar la frecuencia de solicitudes

esquema_aqicn = T.StructType([
    T.StructField('numero_captura', T.IntegerType(), False),
    T.StructField('estacion', T.StringType(), True),
    T.StructField('fecha_medicion', T.StringType(), True),
    T.StructField('momento_captura', T.StringType(), True),
    *[T.StructField(x, T.DoubleType(), True) for x in CLAVES_AQICN.values()]
])
df_aqicn = spark.createDataFrame(filas_aqicn, schema=esquema_aqicn)
fechas_distintas_aqicn = df_aqicn.select('fecha_medicion').distinct().count()
print('AQICN: capturas reales =', df_aqicn.count(), '| instantes de medición distintos =', fechas_distintas_aqicn)
if fechas_distintas_aqicn < CANTIDAD_REQUERIDA:
    print('AVISO DE CALIDAD: las 100 capturas no equivalen a 100 mediciones históricas distintas.')
df_aqicn.show(5, truncate=False)

Estación AQICN: Parque O'Higgins, Chile
Coordenadas: [-33.464142, -70.660797]
AQICN: capturas reales = 100 | instantes de medición distintos = 1
AVISO DE CALIDAD: las 100 capturas no equivalen a 100 mediciones históricas distintas.
+--------------+-----------------------+-------------------------+-------------------------+-----+----+----+----+----+
|numero_captura|estacion               |fecha_medicion           |momento_captura          |PM2_5|PM10|O3  |NO2 |SO2 |
+--------------+-----------------------+-------------------------+-------------------------+-----+----+----+----+----+
|1             |Parque O'Higgins, Chile|2026-11-09T20:00:00-03:00|2026-11-14T22:02:52-03:00|25.0 |29.0|18.4|12.7|NULL|
|2             |Parque O'Higgins, Chile|2026-11-09T20:00:00-03:00|2026-11-14T22:02:55-03:00|25.0 |29.0|18.4|12.7|NULL|
|3             |Parque O'Higgins, Chile|2026-11-09T20:00:00-03:00|2026-11-14T22:02:57-03:00|25.0 |29.0|18.4|12.7|NULL|
|4             |Parque O'Higgins, Chile|2026-11-09T20:

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MCDI502_Sumativa3_Grupo7")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Apache Spark:", spark.version)
print("SparkSession:", spark.sparkContext.appName)
print("SparkContext operativo")

Apache Spark: 4.0.3
SparkSession: MCDI502_Sumativa3_Grupo7
SparkContext operativo


## 4. Ingesta 3/4: MongoDB (ESP32) → `df_mongo_bpm`

Se consulta **MongoDB real**, sin introducir credenciales en el notebook ni depender de conectores JAR. Al ejecutar, proporciona la URI de acceso autorizada por el docente (o configura `MONGO_URI` como variable de entorno). Base y colección por defecto según el ejemplo: `espdb.datos`.

Se leen **hasta 100 documentos**. Si hay menos de 100, el notebook deja el faltante a la vista: la pauta **no permite inferir datos no publicados**. Campos requeridos: `altitudFt` (altura en pies), `presionAtm` y `ppmCO`.

In [23]:

# ============================================================
# SUMATIVA 3 - GRUPO 7
# INGESTA DE DATOS MONGODB - SENSORES ESP32
# ============================================================

import os
from getpass import getpass
from urllib.parse import quote_plus

from pymongo import MongoClient, DESCENDING
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. CONFIGURACIÓN DE APACHE SPARK
# ------------------------------------------------------------

spark = (
    SparkSession.builder
    .appName("MCDI502_Sumativa3_Grupo7")
    .getOrCreate()
)

print("Apache Spark:", spark.version)

# ------------------------------------------------------------
# 2. CONFIGURACIÓN DE MONGODB
# ------------------------------------------------------------

SERVIDOR_MONGO = "mgtt.z8mrzzw.mongodb.net"
BASE_MONGO = "telemetria"
COLECCION_MONGO = "lecturas"
CANTIDAD_REQUERIDA = 100

# Credenciales seguras: no escribir la contraseña en el código
USUARIO_MONGO = os.getenv("MONGO_USER", "USER_MGTT").strip()
CLAVE_MONGO = os.getenv("MONGO_PASSWORD") or getpass(
    "Ingrese la contraseña de MongoDB: "
)

if not USUARIO_MONGO or not CLAVE_MONGO:
    raise RuntimeError("Faltan las credenciales de MongoDB.")

MONGO_URI = (
    f"mongodb+srv://{quote_plus(USUARIO_MONGO)}:"
    f"{quote_plus(CLAVE_MONGO)}@{SERVIDOR_MONGO}/"
    "?appName=MGTT"
)

# ------------------------------------------------------------
# 3. CONEXIÓN A MONGODB ATLAS
# ------------------------------------------------------------

print("\nConectando con MongoDB Atlas...")

cliente_mongo = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=15000,
    connectTimeoutMS=15000
)

try:
    cliente_mongo.admin.command("ping")
    print("Conexión MongoDB establecida correctamente.")

    base_mongo = cliente_mongo[BASE_MONGO]
    coleccion_mongo = base_mongo[COLECCION_MONGO]

    # Contar los documentos reales disponibles
    total_disponible = coleccion_mongo.count_documents({})

    # Extraer las 100 lecturas más recientes del ESP32
    documentos_mongo = list(
        coleccion_mongo.find(
            {},
            {
                "_id": 1,
                "altitudFt": 1,
                "presionAtm": 1,
                "ppmCO": 1,
                "fecha": 1
            }
        )
        .sort("fecha", DESCENDING)
        .limit(CANTIDAD_REQUERIDA)
    )

finally:
    cliente_mongo.close()

# ------------------------------------------------------------
# 4. LIMPIEZA Y CONVERSIÓN DE TIPOS
# ------------------------------------------------------------

def a_numero_mongo(valor):
    if valor is None or str(valor).strip() == "":
        return None

    try:
        return float(str(valor).strip().replace(",", "."))
    except (ValueError, TypeError):
        return None


filas_mongo = []

for documento in documentos_mongo:
    fila = (
        str(documento.get("_id")),
        a_numero_mongo(documento.get("altitudFt")),
        a_numero_mongo(documento.get("presionAtm")),
        a_numero_mongo(documento.get("ppmCO")),
        str(documento.get("fecha") or "")
    )

    filas_mongo.append(fila)

# ------------------------------------------------------------
# 5. DEFINICIÓN DEL ESQUEMA SPARK
# ------------------------------------------------------------

esquema_mongo = T.StructType([
    T.StructField("id_mongo", T.StringType(), True),
    T.StructField("alturaft", T.DoubleType(), True),
    T.StructField("presion_atmosferica", T.DoubleType(), True),
    T.StructField("monoxido_carbono", T.DoubleType(), True),
    T.StructField("fecha_mongo", T.StringType(), True)
])

# ------------------------------------------------------------
# 6. CREACIÓN DEL DATAFRAME REQUERIDO
# ------------------------------------------------------------

df_mongo_bpm = spark.createDataFrame(
    filas_mongo,
    schema=esquema_mongo
)

# ------------------------------------------------------------
# 7. VALIDACIÓN DE LOS 100 REGISTROS
# ------------------------------------------------------------

conteo_mongo = df_mongo_bpm.count()

print("\n========== RESULTADOS MONGODB ==========")
print("Servidor:", SERVIDOR_MONGO)
print("Base de datos:", BASE_MONGO)
print("Colección:", COLECCION_MONGO)
print("Documentos disponibles:", total_disponible)
print("Registros recuperados:", conteo_mongo)

if conteo_mongo == CANTIDAD_REQUERIDA:
    print("CUMPLE: 100 registros obtenidos desde MongoDB.")
else:
    print(
        f"PENDIENTE: se recuperaron {conteo_mongo} "
        f"de {CANTIDAD_REQUERIDA} registros."
    )

# ------------------------------------------------------------
# 8. VERIFICACIÓN DE ESTRUCTURA Y DATOS
# ------------------------------------------------------------

print("\nEsquema del DataFrame:")
df_mongo_bpm.printSchema()

print("\nPrimeros 5 registros:")
df_mongo_bpm.show(5, truncate=False)

print("\nValores nulos por variable:")

df_mongo_bpm.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in [
        "alturaft",
        "presion_atmosferica",
        "monoxido_carbono"
    ]
]).show()

print("\nDataFrame df_mongo_bpm creado correctamente.")


Apache Spark: 4.0.3

Conectando con MongoDB Atlas...
Conexión MongoDB establecida correctamente.

========== RESULTADOS MONGODB ==========
Servidor: mgtt.z8mrzzw.mongodb.net
Base de datos: telemetria
Colección: lecturas
Documentos disponibles: 92744
Registros recuperados: 100
CUMPLE: 100 registros obtenidos desde MongoDB.

Esquema del DataFrame:
root
 |-- id_mongo: string (nullable = true)
 |-- alturaft: double (nullable = true)
 |-- presion_atmosferica: double (nullable = true)
 |-- monoxido_carbono: double (nullable = true)
 |-- fecha_mongo: string (nullable = true)


Primeros 5 registros:
+------------------------+--------+-------------------+----------------+--------------------------+
|id_mongo                |alturaft|presion_atmosferica|monoxido_carbono|fecha_mongo               |
+------------------------+--------+-------------------+----------------+--------------------------+
|6af9097dc1ab6b5957d4e704|831.98  |0.970323           |3.73            |2026-11-15 01:23:09.943000|
|

In [22]:

from pymongo import MongoClient

cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)

try:
    cliente.admin.command("ping")

    base = cliente["telemetria"]

    colecciones = base.list_collection_names()

    print("Base de datos: telemetria")
    print("Colecciones encontradas:", colecciones)

    for nombre in colecciones:
        coleccion = base[nombre]
        cantidad = coleccion.count_documents({})
        ejemplo = coleccion.find_one()

        print(f"\nColección: {nombre}")
        print(f"Total de registros: {cantidad}")

        if ejemplo:
            print("Campos disponibles:")
            print(list(ejemplo.keys()))

            campos_sensor = [
                "altitudFt", "presionAtm", "ppmCO",
                "alturaft", "presion", "co",
                "fecha", "timestamp"
            ]

            print("Valores de sensores encontrados:")
            for campo in campos_sensor:
                if campo in ejemplo:
                    print(f"  {campo}: {ejemplo[campo]}")
        else:
            print("Colección vacía")

finally:
    cliente.close()



Base de datos: telemetria
Colecciones encontradas: ['lecturas']

Colección: lecturas
Total de registros: 92734
Campos disponibles:
['_id', 'temperatura', 'presionPa', 'presionAtm', 'altitudFt', 'ppmCO', 'fecha', '__v']
Valores de sensores encontrados:
  altitudFt: 711.55
  presionAtm: 0.974547
  ppmCO: 5.64
  fecha: 2026-09-28 00:01:56.218000


## 5. Ingesta 4/4: MySQL (ESP32) mediante API REST → `df_mysql_dht`

La fuente se consulta mediante el endpoint PHP/JSON del material docente (origen MySQL), sin inventar una conexión JDBC que la pauta no necesita. La URL puede personalizarse con `URL_MYSQL`. Variables: temperatura interior y humedad. Se toman hasta 100 registros del servicio real.

In [7]:
import os
import requests

URL_MYSQL = os.getenv(
    "URL_MYSQL",
    "http://mgtt.comunidadingenieria.cl/obtener_datos.php"
)

respuesta = requests.get(URL_MYSQL, timeout=35)
respuesta.raise_for_status()
datos_mysql = respuesta.json()
if isinstance(datos_mysql, dict):
    datos_mysql = datos_mysql.get('data') or datos_mysql.get('datos') or datos_mysql.get('resultados')
if not isinstance(datos_mysql, list):
    raise ValueError('MySQL/API REST no devolvió una lista JSON de registros.')
def a_numero_mysql(valor):
    if valor is None or str(valor).strip() == '':
        return None
    try:
        return float(str(valor).strip().replace(',', '.'))
    except (TypeError, ValueError):
        return None
filas_mysql = [(
    str(f.get('id') or f.get('id_mysql') or ''),
    a_numero_mysql(f.get('temperatura', f.get('temperatura_interior'))),
    a_numero_mysql(f.get('humedad')),
    str(f.get('fecha') or ''),
) for f in datos_mysql[:CANTIDAD_REQUERIDA]]
esquema_mysql = T.StructType([
    T.StructField('id_mysql', T.StringType(), True),
    T.StructField('temperatura_interior', T.DoubleType(), True),
    T.StructField('humedad', T.DoubleType(), True),
    T.StructField('fecha_mysql', T.StringType(), True),
])
df_mysql_dht = spark.createDataFrame(filas_mysql, schema=esquema_mysql)
print('MySQL vía API REST:', df_mysql_dht.count(), 'registros reales (se requieren 100).')
df_mysql_dht.show(5, truncate=False)

MySQL vía API REST: 100 registros reales (se requieren 100).
+--------+--------------------+-------+-------------------+
|id_mysql|temperatura_interior|humedad|fecha_mysql        |
+--------+--------------------+-------+-------------------+
|377668  |22.9                |65.5   |2026-11-14 17:09:51|
|377667  |22.9                |65.2   |2026-11-14 17:09:35|
|377666  |23.0                |65.1   |2026-11-14 17:09:18|
|377665  |23.0                |64.9   |2026-11-14 17:09:00|
|377664  |23.1                |64.9   |2026-11-14 17:08:43|
+--------+--------------------+-------+-------------------+
only showing top 5 rows


In [8]:
fuentes_requeridas = [
    "df_openmeteo",
    "df_aqicn",
    "df_mongo_bpm",
    "df_mysql_dht"
]

for nombre in fuentes_requeridas:
    if nombre in globals():
        print(f"✅ {nombre}: disponible")
    else:
        print(f"❌ {nombre}: falta ejecutar su celda")

✅ df_openmeteo: disponible
✅ df_aqicn: disponible
✅ df_mongo_bpm: disponible
✅ df_mysql_dht: disponible


## 6. Comprobación de las cuatro fuentes — ingesta y DataFrames (24 pts)

La rúbrica exige **exactamente 100 registros reales por DataFrame**. Este diagnóstico no rellena filas con valores ficticios. Si alguna fuente queda corta, solicita al docente la fuente o el acceso correcto antes de entregar. Los cuatro esquemas y ejemplos son evidencia requerida.

In [9]:
fuentes = {
    'df_openmeteo': df_openmeteo,
    'df_aqicn': df_aqicn,
    'df_mongo_bpm': df_mongo_bpm,
    'df_mysql_dht': df_mysql_dht,
}
conteos = {nombre: df.count() for nombre, df in fuentes.items()}
for nombre, df in fuentes.items():
    n = conteos[nombre]
    print(f'\n{nombre}: {n}/{CANTIDAD_REQUERIDA} registros —', 'CUMPLE' if n == CANTIDAD_REQUERIDA else 'PENDIENTE')
    df.printSchema()
    df.show(3, truncate=False)
if any(n != CANTIDAD_REQUERIDA for n in conteos.values()):
    print('ATENCIÓN: no están acreditados los 24 puntos de ingesta hasta alcanzar 100 registros en cada fuente.')


df_openmeteo: 100/100 registros — CUMPLE
root
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- uv_index: double (nullable = true)
 |-- fecha_hora: timestamp (nullable = false)

+--------------+--------------------+-------------+--------------+------------------+--------+-------------------+
|temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|fecha_hora         |
+--------------+--------------------+-------------+--------------+------------------+--------+-------------------+
|17.4          |64.0                |0.0          |8.1           |162.0             |0.0     |2026-11-07 23:00:00|
|16.3          |67.0                |0.0          |6.9           |150.0             |0.0     |2026-11-08 01:00:00|
|15.8          |69.0                |0.0          |6

## 7. Preprocesamiento y transformación (24 pts)

- MongoDB: `altura_metros = alturaft * 0.3048` (un pie son 0,3048 metros).
- Open-Meteo: `condicion_clima` con exactamente los cuatro resultados y prioridades definidos en la pauta.

In [10]:
df_mongo_bpm = df_mongo_bpm.withColumn(
    'altura_metros', F.round(F.col('alturaft') * F.lit(0.3048), 4)
)
df_openmeteo = df_openmeteo.withColumn(
    'condicion_clima',
    F.when((F.col('temperature_2m') > 30) & (F.col('relative_humidity_2m') > 70), 'CALOR Y HUMEDAD')
     .when((F.col('temperature_2m') > 30) & (F.col('relative_humidity_2m') <= 70), 'CALOR SECO')
     .when(F.col('temperature_2m') <= 30, 'CLIMA MODERADO')
     .otherwise('CONDICION DESCONOCIDA')
)
print('Altura ft → metros:')
df_mongo_bpm.select('alturaft', 'altura_metros').show(10, truncate=False)
print('Clasificación climática:')
df_openmeteo.select('temperature_2m', 'relative_humidity_2m', 'condicion_clima').show(10, truncate=False)
assert 'altura_metros' in df_mongo_bpm.columns
assert 'condicion_clima' in df_openmeteo.columns

Altura ft → metros:
+--------+-------------+
|alturaft|altura_metros|
+--------+-------------+
+--------+-------------+

Clasificación climática:
+--------------+--------------------+---------------+
|temperature_2m|relative_humidity_2m|condicion_clima|
+--------------+--------------------+---------------+
|17.4          |64.0                |CLIMA MODERADO |
|16.3          |67.0                |CLIMA MODERADO |
|15.8          |69.0                |CLIMA MODERADO |
|14.6          |76.0                |CLIMA MODERADO |
|13.8          |81.0                |CLIMA MODERADO |
|14.0          |79.0                |CLIMA MODERADO |
|23.3          |48.0                |CLIMA MODERADO |
|28.5          |38.0                |CLIMA MODERADO |
|29.6          |40.0                |CLIMA MODERADO |
|30.4          |43.0                |CALOR SECO     |
+--------------+--------------------+---------------+
only showing top 10 rows


## 8. Integración de las cuatro fuentes → `df_unificado` (12 pts)

**Estrategia:** los cuatro sistemas no proporcionan una clave natural compartida. Se ordena cada fuente por su fecha/captura y por identificadores estables, se asigna un `indice` de fila y se usa `full_outer` sobre ese índice.

Esta unión tiene valor demostrativo: **el mismo índice no implica que las mediciones sean simultáneas**. Por eso se conservan fechas y procedencias, y las consultas ambientales se hacen sobre las fuentes originales. Con una fuente de menos de 100 registros, la unión mantiene los datos existentes y muestra nulos para los faltantes.

In [11]:
def con_indice(df, *columnas_orden):
    return df.withColumn('indice', F.row_number().over(Window.orderBy(*[F.col(c).asc_nulls_last() for c in columnas_orden])))

meteo_ind = con_indice(df_openmeteo, 'fecha_hora')
aqicn_ind = con_indice(df_aqicn, 'momento_captura', 'numero_captura')
mongo_ind = con_indice(df_mongo_bpm, 'fecha_mongo', 'id_mongo')
mysql_ind = con_indice(df_mysql_dht, 'fecha_mysql', 'id_mysql')

df_unificado = (meteo_ind
    .join(aqicn_ind, on='indice', how='full_outer')
    .join(mongo_ind, on='indice', how='full_outer')
    .join(mysql_ind, on='indice', how='full_outer')
    .orderBy('indice'))
filas_integradas = df_unificado.count()
print('Integración:', filas_integradas, 'filas; máximo entre las fuentes:', max(conteos.values()))
assert filas_integradas == max(conteos.values()), 'Revisar duplicaciones de índices.'
df_unificado.printSchema()
df_unificado.show(5, truncate=False)

Integración: 100 filas; máximo entre las fuentes: 100
root
 |-- indice: integer (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- uv_index: double (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- condicion_clima: string (nullable = true)
 |-- numero_captura: integer (nullable = true)
 |-- estacion: string (nullable = true)
 |-- fecha_medicion: string (nullable = true)
 |-- momento_captura: string (nullable = true)
 |-- PM2_5: double (nullable = true)
 |-- PM10: double (nullable = true)
 |-- O3: double (nullable = true)
 |-- NO2: double (nullable = true)
 |-- SO2: double (nullable = true)
 |-- id_mongo: string (nullable = true)
 |-- alturaft: double (nullable = true)
 |-- presion_atmosferica: double (nullable = true)
 |-- monoxido_carbono: double (nullable = t

## 9. Spark SQL — vistas temporales y consultas C1–C7 (24 pts)

La rúbrica requiere **cuatro vistas temporales**, una por fuente, y **siete consultas SQL** con sus tablas visibles. Se ejecutan las consultas sobre los datos de origen, más las columnas calculadas de la sección 7.

In [12]:
for nombre, df in {
    'df_openmeteo': df_openmeteo,
    'df_aqicn': df_aqicn,
    'df_mongo_bpm': df_mongo_bpm,
    'df_mysql_dht': df_mysql_dht,
}.items():
    df.createOrReplaceTempView(nombre)
    print('Vista temporal:', nombre)

Vista temporal: df_openmeteo
Vista temporal: df_aqicn
Vista temporal: df_mongo_bpm
Vista temporal: df_mysql_dht


### C1 — Diez temperaturas más altas de Open-Meteo

In [13]:
c1 = spark.sql('''
SELECT fecha_hora, temperature_2m
FROM df_openmeteo
ORDER BY temperature_2m DESC NULLS LAST, fecha_hora ASC
LIMIT 10
''')
c1.show(10, truncate=False)

+-------------------+--------------+
|fecha_hora         |temperature_2m|
+-------------------+--------------+
|2026-11-12 14:00:00|31.3          |
|2026-11-12 16:00:00|31.0          |
|2026-11-12 13:00:00|30.7          |
|2026-11-08 14:00:00|30.4          |
|2026-11-11 15:00:00|30.2          |
|2026-11-08 12:00:00|29.6          |
|2026-11-12 18:00:00|29.1          |
|2026-11-12 11:00:00|28.7          |
|2026-11-08 16:00:00|28.6          |
|2026-11-08 11:00:00|28.5          |
+-------------------+--------------+



### C2 — Humedad relativa promedio para un día y mes elegidos

Se escoge el día calendario correspondiente a la última observación, junto con el año para evitar mezclas entre años. El cálculo se hace con la muestra de 100 registros.

In [14]:
fecha_objetivo = df_openmeteo.agg(F.max(F.to_date('fecha_hora'))).first()[0]
print('Día/mes/año consultado:', fecha_objetivo)
c2 = spark.sql(f'''
SELECT DATE(fecha_hora) AS dia, COUNT(relative_humidity_2m) AS observaciones,
       ROUND(AVG(relative_humidity_2m), 2) AS humedad_media
FROM df_openmeteo
WHERE DATE(fecha_hora) = DATE '{fecha_objetivo}'
GROUP BY DATE(fecha_hora)
''')
c2.show(truncate=False)

Día/mes/año consultado: 2026-11-14
+----------+-------------+-------------+
|dia       |observaciones|humedad_media|
+----------+-------------+-------------+
|2026-11-14|14           |87.14        |
+----------+-------------+-------------+



### C3 — Cantidad de registros con precipitación superior a 2 mm durante la última semana

Se cuenta sobre las 100 observaciones seleccionadas de la última semana (no sobre todas las horas posibles).

In [15]:
c3 = spark.sql('''
SELECT COUNT(*) AS registros_precipitacion_mayor_2mm
FROM df_openmeteo
WHERE precipitation > 2
''')
c3.show(truncate=False)

+---------------------------------+
|registros_precipitacion_mayor_2mm|
+---------------------------------+
|0                                |
+---------------------------------+



### C4 — Promedio de PM2.5, PM10, O3, NO2 y SO2 (valores IAQI)

In [16]:
c4 = spark.sql('''
SELECT ROUND(AVG(PM2_5), 2) AS PM2_5_promedio,
       ROUND(AVG(PM10), 2) AS PM10_promedio,
       ROUND(AVG(O3), 2) AS O3_promedio,
       ROUND(AVG(NO2), 2) AS NO2_promedio,
       ROUND(AVG(SO2), 2) AS SO2_promedio
FROM df_aqicn
''')
c4.show(truncate=False)

+--------------+-------------+-----------+------------+------------+
|PM2_5_promedio|PM10_promedio|O3_promedio|NO2_promedio|SO2_promedio|
+--------------+-------------+-----------+------------+------------+
|25.0          |29.0         |18.4       |12.7        |NULL        |
+--------------+-------------+-----------+------------+------------+



### C5 — Registros de AQICN con NO2 superior a 4

In [17]:
c5 = spark.sql('''
SELECT COUNT(*) AS registros_NO2_mayor_4
FROM df_aqicn
WHERE NO2 > 4
''')
c5.show(truncate=False)

+---------------------+
|registros_NO2_mayor_4|
+---------------------+
|100                  |
+---------------------+



### C6 — Diez registros de MongoDB con `altura_metros`

In [25]:
from pyspark.sql import functions as F

# 1. Convertir altura de pies a metros
df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros",
    F.round(F.col("alturaft") * 0.3048, 4)
)

# 2. Actualizar la vista temporal para Spark SQL
df_mongo_bpm.createOrReplaceTempView("df_mongo_bpm")

# 3. Consulta C6
c6 = spark.sql("""
    SELECT
        id_mongo,
        fecha_mongo,
        alturaft,
        altura_metros,
        presion_atmosferica,
        monoxido_carbono
    FROM df_mongo_bpm
    ORDER BY fecha_mongo ASC NULLS LAST, id_mongo ASC
    LIMIT 10
""")

# 4. Mostrar resultados
print("C6 - Primeros 10 registros de MongoDB")
c6.show(10, truncate=False)


C6 - Primeros 10 registros de MongoDB
+------------------------+--------------------------+--------+-------------+-------------------+----------------+
|id_mongo                |fecha_mongo               |alturaft|altura_metros|presion_atmosferica|monoxido_carbono|
+------------------------+--------------------------+--------+-------------+-------------------+----------------+
|6af902eec1ab6b5957d4e63e|2026-11-15 00:55:10.989000|838.7   |255.6358     |0.970057           |3.65            |
|6af902ffc1ab6b5957d4e640|2026-11-15 00:55:27.842000|837.86  |255.3797     |0.970077           |3.65            |
|6af90310c1ab6b5957d4e642|2026-11-15 00:55:44.913000|839.26  |255.8064     |0.970047           |3.58            |
|6af90322c1ab6b5957d4e644|2026-11-15 00:56:02.816000|837.58  |255.2944     |0.970106           |3.62            |
|6af90333c1ab6b5957d4e646|2026-11-15 00:56:19.653000|838.14  |255.4651     |0.970106           |3.65            |
|6af90344c1ab6b5957d4e648|2026-11-15 00:56:36.4760

### C7 — Diez registros de Open-Meteo con `condicion_clima`

In [19]:
c7 = spark.sql('''
SELECT fecha_hora, temperature_2m, relative_humidity_2m, condicion_clima
FROM df_openmeteo
ORDER BY fecha_hora ASC
LIMIT 10
''')
c7.show(10, truncate=False)

+-------------------+--------------+--------------------+---------------+
|fecha_hora         |temperature_2m|relative_humidity_2m|condicion_clima|
+-------------------+--------------+--------------------+---------------+
|2026-11-07 23:00:00|17.4          |64.0                |CLIMA MODERADO |
|2026-11-08 01:00:00|16.3          |67.0                |CLIMA MODERADO |
|2026-11-08 02:00:00|15.8          |69.0                |CLIMA MODERADO |
|2026-11-08 04:00:00|14.6          |76.0                |CLIMA MODERADO |
|2026-11-08 06:00:00|13.8          |81.0                |CLIMA MODERADO |
|2026-11-08 07:00:00|14.0          |79.0                |CLIMA MODERADO |
|2026-11-08 09:00:00|23.3          |48.0                |CLIMA MODERADO |
|2026-11-08 11:00:00|28.5          |38.0                |CLIMA MODERADO |
|2026-11-08 12:00:00|29.6          |40.0                |CLIMA MODERADO |
|2026-11-08 14:00:00|30.4          |43.0                |CALOR SECO     |
+-------------------+--------------+--

## 10. Controles finales, resultados y limitaciones

La comprobación final muestra el estado de cada fuente, el DataFrame integrado, las columnas y las consultas. **No marca como cumplido un requisito que los datos no respaldan**. Si MongoDB/API no provee 100 registros, solicita acceso o registros adicionales a tu docente antes de la entrega.

In [27]:
fuentes = {
    "df_openmeteo": df_openmeteo,
    "df_aqicn": df_aqicn,
    "df_mongo_bpm": df_mongo_bpm,
    "df_mysql_dht": df_mysql_dht
}

# Recalcular cantidades
conteos = {
    nombre: df.count()
    for nombre, df in fuentes.items()
}

print("========== RESUMEN DE VALIDACIONES ==========\n")

for nombre, cantidad in conteos.items():
    estado = "OK" if cantidad == 100 else "FALTAN REGISTROS"
    print(f"{nombre:18s}: {cantidad:3d} filas - {estado}")

print("\n========== TRANSFORMACIONES ==========")

print(
    "altura_metros:",
    "OK" if "altura_metros" in df_mongo_bpm.columns
    else "PENDIENTE"
)

print(
    "condicion_clima:",
    "OK" if "condicion_clima" in df_openmeteo.columns
    else "PENDIENTE"
)

print("\n========== INTEGRACIÓN ==========")

if "df_unificado" in globals():
    print("df_unificado:", df_unificado.count(), "filas")
    print("Nota: ejecutar nuevamente la integración si cambió alguna fuente.")
else:
    print("df_unificado: PENDIENTE")

print("\n========== CONSULTAS SQL ==========")

for nombre in ["c1", "c2", "c3", "c4", "c5", "c6", "c7"]:
    if nombre in globals():
        resultado = globals()[nombre]
        print(f"{nombre}: {resultado.count()} filas")
    else:
        print(f"{nombre}: PENDIENTE")

print("\n========== RESULTADO FINAL ==========")

if all(cantidad == 100 for cantidad in conteos.values()):
    print("CUMPLE: 4/4 fuentes contienen 100 registros.")
else:
    print("PENDIENTE: existen fuentes con menos de 100 registros.")


========== RESUMEN DE VALIDACIONES ==========

df_openmeteo      : 100 filas - OK
df_aqicn          : 100 filas - OK
df_mongo_bpm      : 100 filas - OK
df_mysql_dht      : 100 filas - OK

========== TRANSFORMACIONES ==========
altura_metros: OK
condicion_clima: OK

========== INTEGRACIÓN ==========
df_unificado: 100 filas
Nota: ejecutar nuevamente la integración si cambió alguna fuente.

========== CONSULTAS SQL ==========
c1: 10 filas
c2: 1 filas
c3: 1 filas
c4: 1 filas
c5: 1 filas
c6: 10 filas
c7: 10 filas

========== RESULTADO FINAL ==========
CUMPLE: 4/4 fuentes contienen 100 registros.


## 11. Presentación y entrega

**Video obligatorio:** 7–10 minutos con participación de los tres integrantes (Cristian, Felipe y Jorge), explicando código, decisiones de conexión, transformación, unión y salidas SQL.

**Nombre notebook:** `mcdi502_s3_grupo7.ipynb`. **Video sugerido por pauta:** `mcdi502_s3_grupo7_video.mp4` (el ejemplo oficial supera su límite genérico de 20 caracteres; ante la contradicción, priorizar el nombre expresamente indicado o consultar al docente). **Entrega:** ambos archivos en un único ZIP en Canvas.

Checklist previo: (1) ejecutar en entorno nuevo con acceso a las cuatro fuentes, (2) verificar los 100 registros por fuente y ciudad AQICN, (3) guardar salidas sin errores, (4) grabar video y (5) comprimir y entregar. **No incluir credenciales ni la URI MongoDB en el ZIP.**